# Семинар 4 — логистическая регрессия

На лекции мы получили цепочку

$$
X,\beta
\longrightarrow
z=X\beta
\longrightarrow
p=\sigma(z)
\longrightarrow
R(\beta)
\longrightarrow
\nabla R(\beta).
$$

Сегодня соберём эту цепочку в NumPy и обучим модель тем же градиентным спуском, который использовали на прошлом семинаре.

После занятия должны быть понятны четыре вещи:

1. как линейный score превращается в вероятность;
2. как считается log loss;
3. откуда в коде появляется градиент $\frac{1}{n}X^\top(p-y)$;
4. чем отличаются score, вероятность и итоговая метка класса.


## 0. Данные

Рассматриваем задачу оттока клиентов сервиса.

В таблице три столбца:

- `weekly_sessions_change` — изменение числа сессий за последнюю неделю относительно предыдущей;
- `support_requests_30d` — число обращений в поддержку за последние 30 дней;
- `churn_30d` — целевая переменная: `1`, если клиент ушёл в течение следующих 30 дней, и `0` в противном случае.

Данные синтетические. Они нужны для прозрачной двумерной картинки и полностью воспроизводимого обучения.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

DATA_PATH = Path("seminar_week4_data_v6.csv")


Сначала загрузим таблицу и посмотрим на сами данные.


In [ ]:
data = pd.read_csv(DATA_PATH)
data.head(8)


Проверим размер таблицы и число объектов каждого класса.


In [ ]:
print("Размер таблицы:", data.shape)
data["churn_30d"].value_counts().sort_index().rename("число объектов")


Поскольку признаков два, выборку можно сразу показать на плоскости.


In [ ]:
plt.figure(figsize=(7.2, 4.8))

for cls, marker, label in [
    (0, "o", "класс 0: клиент остался"),
    (1, "^", "класс 1: клиент ушёл"),
]:
    part = data[data["churn_30d"] == cls]
    plt.scatter(
        part["weekly_sessions_change"],
        part["support_requests_30d"],
        marker=marker,
        label=label,
        alpha=0.8,
    )

plt.xlabel("Изменение числа сессий за неделю")
plt.ylabel("Обращения в поддержку за 30 дней")
plt.legend()
plt.show()


### Делим данные

`train_test_split` умеет работать непосредственно с `DataFrame` и `Series`, поэтому на этом шаге NumPy нам не нужен.

Параметр `stratify=y` сохраняет примерно ту же долю классов в обеих частях выборки.


In [ ]:
feature_names = ["weekly_sessions_change", "support_requests_30d"]
target_name = "churn_30d"

data.index.name = "row_id"

X_df = data[feature_names]
y_series = data[target_name]

X_train_df, X_valid_df, y_train_series, y_valid_series = train_test_split(
    X_df,
    y_series,
    test_size=0.25,
    random_state=42,
    stratify=y_series,
)


`row_id` — это номер строки в исходной таблице. `train_test_split` выбирает строки в случайном порядке, но **не сбрасывает исходный индекс**.

Поэтому в validation дальше могут встречаться индексы вроде `53`, `119`, `7` — это не номера позиций внутри validation, а идентификаторы исходных строк. Мы специально сохраняем их, чтобы один и тот же объект можно было проследить через разные таблицы.

Важно и дальше: split выполняется **один раз**. Ручная модель и scikit-learn будут обучаться на одних и тех же `X_train_df, y_train_series` и сравниваться на одних и тех же `X_valid_df, y_valid_series`.

In [ ]:
pd.DataFrame({
    "часть": ["train", "valid"],
    "число объектов": [len(X_train_df), len(X_valid_df)],
    "доля класса 1": [y_train_series.mean(), y_valid_series.mean()],
})


### Переходим к матрицам для ручных вычислений

Дальше будем сами вычислять $X\beta$ и $X^\top(p-y)$. Для этих операций удобно использовать NumPy-массивы.

`to_numpy(dtype=float)` здесь выполняет две задачи:

- извлекает численные значения из `DataFrame` и `Series`;
- приводит матрицу признаков к вещественному типу.

Для самого `train_test_split` эта операция не требовалась.


In [ ]:
X_train_features = X_train_df.to_numpy(dtype=float)
X_valid_features = X_valid_df.to_numpy(dtype=float)

y_train = y_train_series.to_numpy(dtype=float)
y_valid = y_valid_series.to_numpy(dtype=float)

print("X_train_features:", X_train_features.shape)
print("y_train:", y_train.shape)


### Столбец единиц и размерности

У модели есть свободный член:

$$
z_i=\beta_0+\beta_1x_{i1}+\beta_2x_{i2}.
$$

Чтобы записать это одним матричным произведением, добавим к каждому объекту первый признак, равный единице:

$$
x_i=
\begin{bmatrix}
1\\
x_{i1}\\
x_{i2}
\end{bmatrix},
\qquad
\beta=
\begin{bmatrix}
\beta_0\\
\beta_1\\
\beta_2
\end{bmatrix}.
$$

После этого:

- два исходных признака дают матрицу размера $n\times2$;
- после добавления столбца единиц получаем $X\in\mathbb R^{n\times3}$;
- вектор параметров имеет размер $\beta\in\mathbb R^3$;
- произведение $X\beta$ содержит один score на каждый объект и имеет размер $n$.


In [ ]:
ones_train = np.ones((len(X_train_features), 1))
ones_valid = np.ones((len(X_valid_features), 1))

X_train = np.column_stack([ones_train, X_train_features])
X_valid = np.column_stack([ones_valid, X_valid_features])

print("До столбца единиц:", X_train_features.shape)
print("После столбца единиц:", X_train.shape)


In [ ]:
pd.DataFrame(
    X_train[:5],
    columns=["1", *feature_names],
)


## 1. Сигмоида

Логистическая регрессия сначала вычисляет линейный score

$$
z=X\beta,
$$

а затем переводит его в вероятность положительного класса:

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Начнём с обычных значений score и реализуем эту формулу напрямую.


### Задание 1. Реализуйте `sigmoid`


In [ ]:
def sigmoid(z):
    """
    Вычисляет сигмоиду поэлементно.

    Parameters
    ----------
    z : array-like
        Линейные score. Допустим скаляр, вектор или массив любой формы.

    Returns
    -------
    p : np.ndarray
        Значения сигмоиды той же формы, что и входной массив z.
        Математически при конечном z значение лежит строго между 0 и 1.
        В float64 на экстремальных score результат может округлиться до 0.0 или 1.0.
    """
    z = np.asarray(z, dtype=float)

    # TODO: реализуйте сигмоиду
    return ...


Проверим функцию на нескольких значениях score.


In [ ]:
z_demo = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])
p_demo = sigmoid(z_demo)

pd.DataFrame({
    "z": z_demo,
    "sigmoid(z)": p_demo,
})


In [ ]:
assert p_demo.shape == z_demo.shape
assert np.isclose(sigmoid(np.array([0.0]))[0], 0.5)
assert np.all(np.diff(p_demo) > 0)


На этих значениях хорошо видны три свойства: сигмоида монотонно возрастает, при $z=0$ даёт $0.5$, а большие по модулю score приближают вероятность к 0 или 1.


### Практическая оговорка: очень большие по модулю score

Формула

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

полностью корректна математически. Но математическая формула и способ её вычисления на компьютере — не одно и то же.

Специально возьмём экстремальные score $z=\pm1000$. Для $z=-1000$ в знаменателе прямой формулы нужно вычислить $e^{1000}$. Посмотрим, что произойдёт в `float64`.

In [ ]:
z_extreme = np.array([-1000.0, 1000.0])

# Для z = -1000 здесь нужно вычислить exp(1000).
exp_minus_z = np.exp(-z_extreme)

pd.DataFrame({
    "z": z_extreme,
    "exp(-z)": exp_minus_z,
})

Для $z=-1000$ NumPy получил `inf`: промежуточное число $e^{1000}$ не помещается в `float64`.

Важно: **сама сигмоида от этого не перестала быть корректной математически**. Проблема возникла раньше — в буквальном вычислении промежуточного $e^{-z}$.

Теперь доведём нашу прямую формулу до конца.

In [ ]:
p_naive_extreme = 1.0 / (1.0 + exp_minus_z)

pd.DataFrame({
    "z": z_extreme,
    "наша прямая sigmoid": p_naive_extreme,
})

Итог выглядит правдоподобно: получились `0.0` и `1.0`. Но для $z=-1000$ вычисление уже прошло через `inf`.

Более того, математически при любом конечном $z$

$$
0<\sigma(z)<1.
$$

Значения `0.0` и `1.0` здесь появились из-за конечной точности `float64`: очень маленькое отличие от 0 или 1 компьютер уже не хранит.

Чтобы не создавать гигантское промежуточное число, ту же сигмоиду можно вычислять устойчивее. Например,

$$
\sigma(z)=
\begin{cases}
\dfrac{1}{1+e^{-z}}, & z\ge 0,\\[6pt]
\dfrac{e^z}{1+e^z}, & z<0.
\end{cases}
$$

При отрицательном $z$ во второй записи вычисляется маленькое $e^z$, а не огромное $e^{-z}$.

`scipy.special.expit` — готовая численно устойчивая реализация сигмоиды

In [ ]:
from scipy.special import expit

p_expit_extreme = expit(z_extreme)

pd.DataFrame({
    "z": z_extreme,
    "наша прямая sigmoid": p_naive_extreme,
    "scipy.special.expit": p_expit_extreme,
})

В этой ячейке предупреждения об overflow уже нет: `expit` вычисляет ту же функцию устойчивым способом.

На настолько экстремальных score обе реализации всё равно печатают `0.0` и `1.0`, потому что это уже предел точности `float64`. Именно это наблюдение понадобится через несколько ячеек: если затем буквально подставить такие числа в $\log p$ или $\log(1-p)$, возникнет новая численная проблема.

Общего ограничения вида $|z|<c$ у логистической регрессии нет. Score может быть любым вещественным числом. Для обычных учебных вычислений наша простая `sigmoid` достаточна; в библиотечном коде разумно использовать готовую устойчивую реализацию.

## 2. Log loss

Для одного объекта:

$$
L(y,p)=-y\log p-(1-y)\log(1-p).
$$

При $y=1$ остаётся $-\log p$. Сначала посмотрим, как меняется штраф для нескольких прогнозов.


In [ ]:
p_examples = np.array([0.9, 0.5, 0.01])

pd.DataFrame({
    "p при y=1": p_examples,
    "-log(p)": -np.log(p_examples),
})


Теперь проверим буквальную формулу на значениях $p=0$ и $p=1$.

Возьмём два **идеальных** прогноза:

- для объекта класса 0 модель дала $p=0$;
- для объекта класса 1 модель дала $p=1$.

Содержательно log loss у обоих должна быть равна нулю. Посмотрим, что получится, если NumPy буквально вычислит обе части формулы

$$
-y\log p-(1-y)\log(1-p).
$$

In [ ]:
y_edge = np.array([0.0, 1.0])
p_edge = np.array([0.0, 1.0])

term_y1 = -y_edge * np.log(p_edge)
term_y0 = -(1.0 - y_edge) * np.log(1.0 - p_edge)
loss_edge_naive = term_y1 + term_y0

pd.DataFrame({
    "y": y_edge,
    "p": p_edge,
    "-y log(p)": term_y1,
    "-(1-y) log(1-p)": term_y0,
    "наивная log loss": loss_edge_naive,
})

Получили `nan`, хотя прогнозы идеальные.

Причина не в определении log loss, а в буквальном floating-point вычислении. Например, при $y=0$ и $p=0$ NumPy всё равно считает первое слагаемое

$$
-0\cdot\log 0.
$$

Здесь $\log 0=-\infty$, а произведение `0 * (-inf)` в машинной арифметике даёт `nan`. Математически соответствующий вклад понимается через предел и равен нулю, но NumPy этот предел за нас не берёт.

Для нашей учебной реализации нужен простой способ **не передавать в логарифм ровно 0 или 1**. Используем `np.clip`.

In [ ]:
p_demo = np.array([0.0, 0.00001, 0.2, 0.8, 0.99999, 1.0])
eps_demo = 0.001  # нарочно крупное значение, чтобы действие было видно

p_demo_safe = np.clip(p_demo, eps_demo, 1.0 - eps_demo)

pd.DataFrame({
    "исходное p": p_demo,
    "после np.clip": p_demo_safe,
})

`np.clip(p, eps, 1 - eps)` действует поэлементно:

- значения ниже `eps` становятся равны `eps`;
- значения выше `1 - eps` становятся равны `1 - eps`;
- значения внутри интервала остаются прежними.

В самой функции возьмём гораздо меньшее `eps=1e-12`. Оно не является гиперпараметром модели.

### Задание 2. Реализуйте `binary_log_loss`


In [ ]:
def binary_log_loss(y, p, eps=1e-12):
    """
    Вычисляет среднюю binary log loss по набору объектов.

    Parameters
    ----------
    y : array-like, shape (n,)
        Истинные бинарные метки 0/1.
    p : array-like, shape (n,)
        Оценённые вероятности положительного класса.
    eps : float, default=1e-12
        Малое положительное число для защиты log(p) и log(1-p)
        от вычисления log(0).

    Returns
    -------
    loss : float
        Среднее значение log loss по n объектам.
    """
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    p_safe = ...
    losses = ...

    return ...


Проверим несколько свойств функции и сверим результат с `sklearn.metrics.log_loss`.


In [ ]:
y_check = np.array([1.0, 1.0, 0.0, 0.0])
p_check = np.array([0.9, 0.6, 0.2, 0.05])

our_loss = binary_log_loss(y_check, p_check)
sklearn_loss = log_loss(y_check, p_check)

print("Наша функция:", our_loss)
print("sklearn:", sklearn_loss)
print("Совпадают:", np.isclose(our_loss, sklearn_loss))


In [ ]:
assert binary_log_loss(np.array([1.0]), np.array([0.9])) < binary_log_loss(np.array([1.0]), np.array([0.5]))
assert binary_log_loss(np.array([1.0]), np.array([0.01])) > 4.0
assert np.isclose(our_loss, sklearn_loss)


edge_loss = binary_log_loss(np.array([0.0, 1.0]), np.array([0.0, 1.0]))
assert np.isfinite(edge_loss)
assert edge_loss < 1e-9

### Эквивалентная запись через score

Clipping решает конкретную проблему нашей учебной реализации: мы сначала вычисляем вероятность, а затем буквально берём $\log p$ и $\log(1-p)$.

Но есть более устойчивый путь. Если подставить $p=\sigma(z)$ и упростить выражение, ту же loss можно записать непосредственно через score:

$$
L(y,z)=\log(1+e^z)-yz.
$$

Для устойчивого вычисления выражений вида $\log(1+e^z)$ в численных библиотеках используют `logaddexp` или `softplus`.

Поэтому в практических библиотеках часто стараются считать loss **из score/logits напрямую**, не проходя через уже округлённые вероятности `0.0` и `1.0`. `np.clip` нам нужен именно как простой и прозрачный учебный приём для вероятностной формулы.

Практически встречаются оба подхода:

- если функция получает **уже готовые вероятности**, их нередко защищают от точных 0 и 1 clipping-ом или специальной устойчивой реализацией логарифма;
- при **обучении модели** чаще выгоднее работать прямо со score/logits и считать cross-entropy устойчивой формулой без промежуточных `log(sigmoid(z))`.

`softplus` — это название функции

$$
\operatorname{softplus}(z)=\log(1+e^z).
$$

А `np.logaddexp(a, b)` устойчиво вычисляет

$$
\log(e^a+e^b).
$$

Поэтому

```python
np.logaddexp(0.0, z)
```

— это численно устойчивая реализация `softplus(z)`. В PyTorch, TensorFlow и JAX `softplus` встречается как отдельная функция; в NumPy тот же объект удобно считать через `logaddexp`.

## 3. Один шаг градиентного спуска

На лекции для одного объекта получили

$$
\nabla_\beta L_i=(p_i-y_i)x_i.
$$

Перед кодом разложим эту формулу на два звена chain rule:

$$
\frac{\partial L_i}{\partial z_i}=p_i-y_i,
\qquad
\nabla_\beta z_i=x_i.
$$

Поэтому

$$
\nabla_\beta L_i
=
\frac{\partial L_i}{\partial z_i}\nabla_\beta z_i
=
(p_i-y_i)x_i.
$$

То есть `p_i - y_i` — одно число: как loss меняется при изменении score. Вектор $x_i$ показывает, как score зависит от каждого коэффициента. Их произведение даёт по одной производной на каждый параметр модели.

Разберём это на четырёх объектах. После этого матричная запись $X^\top(p-y)$ будет просто короткой записью тех же вычислений.

In [ ]:
small = pd.DataFrame({
    "weekly_sessions_change": [-3.0, -1.0, 1.0, 3.0],
    "support_requests_30d": [4.0, 3.0, 1.0, 0.0],
    "churn_30d": [1, 1, 0, 0],
})

small


Добавим столбец единиц, чтобы свободный член $\beta_0$ входил в тот же вектор параметров.


Здесь мы сознательно повторяем уже знакомую операцию на маленькой выборке: первый столбец единиц отвечает за свободный член.

In [ ]:
X_small_features = small[feature_names].to_numpy(dtype=float)
X_small = np.column_stack([np.ones(len(small)), X_small_features])
y_small = small["churn_30d"].to_numpy(dtype=float)

print("X_small:", X_small.shape)
print("y_small:", y_small.shape)

pd.DataFrame(
    X_small,
    columns=["1", *feature_names],
)


Начнём с нулевых параметров:

$$
\beta^{(0)}=(0,0,0).
$$

Тогда каждый score равен нулю, поэтому каждый объект получает вероятность $0.5$.


В коде назовём весь стартовый вектор `beta_start`, чтобы не путать номер итерации с отдельными коэффициентами $\beta_0,\beta_1,\beta_2$.

In [ ]:
beta_start = np.zeros(3)

z_start = X_small @ beta_start
p_start = sigmoid(z_start)

pd.DataFrame({
    "y": y_small,
    "score z": z_start,
    "p": p_start,
})

Сначала появляется производная loss по score:

$$
\frac{\partial L_i}{\partial z_i}=p_i-y_i.
$$

В коде назовём вектор этих производных `dL_dz_start`: одна величина $\partial L_i/\partial z_i$ на каждый объект в стартовой точке.

Для положительного объекта при $p_i=0.5$ получаем $-0.5$, для отрицательного — $+0.5$.

In [ ]:
dL_dz_start = p_start - y_small

pd.DataFrame({
    "y": y_small,
    "p": p_start,
    "p - y": dL_dz_start,
})

Теперь проследим **первый объект отдельно**.

У него

$$
x_1=
\begin{bmatrix}
1\\-3\\4
\end{bmatrix},
\qquad
 y_1=1,
\qquad
 p_1=0.5.
$$

Поэтому

$$
p_1-y_1=-0.5,
$$

и градиент loss этого объекта по трём параметрам равен

$$
\nabla_\beta L_1
=-0.5
\begin{bmatrix}
1\\-3\\4
\end{bmatrix}
=
\begin{bmatrix}
-0.5\\1.5\\-2.0
\end{bmatrix}.
$$

Три числа — это частные производные по $\beta_0$, $\beta_1$ и $\beta_2$.


In [ ]:
x_first = X_small[0]
dL_dz_first = dL_dz_start[0]
grad_first = dL_dz_first * x_first

# Ниже только оформляем результат как Series для удобного чтения.
pd.Series(
    grad_first,
    index=["dL1/dbeta0", "dL1/dbeta1", "dL1/dbeta2"],
)

Отдельно про свободный член. Для него $x_{i0}=1$, поэтому для отдельного объекта

$$
\frac{\partial L_i}{\partial \beta_0}=p_i-y_i.
$$

Эта производная **не обязана быть равна нулю**.

В нашем маленьком примере в стартовой точке $p_i=0.5$ для всех четырёх объектов, а классы сбалансированы: две единицы и два нуля. Поэтому после усреднения вкладов градиент по свободному члену случайно получается нулевым. Это свойство именно этой точки и этой мини-выборки, а не общее правило.

Для нерегуляризованной логистической регрессии в конечной точке оптимума средний градиент по intercept тоже должен быть близок к нулю — уже как условие оптимума.

То же вычисление нужно выполнить для всех четырёх объектов.

`dL_dz_start` имеет форму `(4,)`: по одному числу $\partial L_i/\partial z_i$ на объект. Матрица `X_small` имеет форму `(4, 3)`: четыре объекта и три компоненты $(1,x_1,x_2)$.

Нам нужно сделать для каждой строки одно и то же действие:

```text
строка X_small[0] × dL_dz_start[0]
строка X_small[1] × dL_dz_start[1]
строка X_small[2] × dL_dz_start[2]
строка X_small[3] × dL_dz_start[3]
```

Почему нельзя просто написать

```python
X_small * dL_dz_start
```

при формах `(4, 3)` и `(4,)`?

NumPy сравнивает размеры справа налево. Поэтому он попробует совместить последний размер матрицы `3` с единственным размером вектора `4`. Размеры `3` и `4` несовместимы, и такое умножение даст `ValueError`.

Для сравнения, вектор формы `(3,)` здесь бы сработал: NumPy воспринял бы его как три множителя **по столбцам** и применил один и тот же вектор ко всем четырём строкам.

Нам нужно другое — по одному множителю **на строку**. Поэтому явно превращаем `(4,)` в столбец `(4, 1)`:

```python
dL_dz_start[:, None]
```

Теперь формы согласуются:

- `X_small`: `(4, 3)`;
- `dL_dz_start[:, None]`: `(4, 1)`.

В результате каждая строка `X_small` умножается на своё число, а итог снова имеет форму `(4, 3)`.

In [ ]:
print("X_small.shape:", X_small.shape)
print("dL_dz_start.shape:", dL_dz_start.shape)
print("dL_dz_start[:, None].shape:", dL_dz_start[:, None].shape)

In [ ]:
dL_dbeta_per_object = X_small * dL_dz_start[:, None]

# Ниже только собираем удобную таблицу для просмотра.
# Эти строки не являются отдельным шагом алгоритма.
gradient_table = small.copy()
gradient_table["p - y"] = dL_dz_start
gradient_table["∂L/∂β0"] = dL_dbeta_per_object[:, 0]
gradient_table["∂L/∂β1"] = dL_dbeta_per_object[:, 1]
gradient_table["∂L/∂β2"] = dL_dbeta_per_object[:, 2]

gradient_table

Последние три столбца нужно читать построчно. Например, первая строка снова даёт

$$
\nabla_\beta L_1=(-0.5,\;1.5,\;-2.0),
$$

который мы только что посчитали отдельно. Остальные строки — такие же градиенты для остальных объектов.

Здесь таблица полезнее отдельного графика: пространство параметров трёхмерное, а задача этого блока — увидеть, как формула одного объекта превращается в матричное вычисление для всей выборки.

Каждая строка последних трёх столбцов — отдельный вектор

$$
\nabla_\beta L_i
=
\begin{bmatrix}
\partial L_i/\partial\beta_0\\
\partial L_i/\partial\beta_1\\
\partial L_i/\partial\beta_2
\end{bmatrix}.
$$

Эмпирический риск — средняя loss:

$$
R(\beta)=\frac1n\sum_{i=1}^nL_i,
$$

поэтому его градиент получается усреднением этих строк:

$$
\nabla R(\beta)
=\frac1n\sum_{i=1}^n\nabla L_i.
$$

Та же операция записывается матрично:

$$
\nabla R(\beta)=\frac1nX^\top(p-y).
$$

Здесь $X^\top$ имеет размер $3\times4$, а $(p-y)$ — длину 4. Результат имеет длину 3: по одной производной на каждый параметр модели.


В коде это усреднение будет записано как

```python
dL_dbeta_per_object.mean(axis=0)
```

Почему `axis=0`? В матрице формы `(4, 3)` строки — объекты, а столбцы — параметры. Усредняем **по объектам**, то есть по строкам, и сохраняем три столбца. В результате получаем вектор длины 3 — градиент риска по трём параметрам.

In [ ]:
grad_R_start = dL_dbeta_per_object.mean(axis=0)

pd.Series(
    grad_R_start,
    index=["dR/dbeta0", "dR/dbeta1", "dR/dbeta2"],
)

Проверим на этих четырёх объектах, что усреднение градиентов объектов и матричная формула дают один и тот же результат.

In [ ]:
grad_R_start_matrix = X_small.T @ dL_dz_start / len(y_small)

pd.DataFrame({
    "параметр": ["beta0", "beta1", "beta2"],
    "среднее градиентов объектов": grad_R_start,
    "X.T @ (p - y) / n": grad_R_start_matrix,
})

Знаки можно связать с данными.

- $\frac{\partial R}{\partial\beta_1}>0$, поэтому шаг по антиградиенту уменьшит $\beta_1$. В этой маленькой выборке снижение недельной активности связано с классом 1.
- $\frac{\partial R}{\partial\beta_2}<0$, поэтому шаг по антиградиенту увеличит $\beta_2$. Большее число обращений в поддержку связано с классом 1.
- Производная по свободному члену равна нулю: на стартовом шаге положительные и отрицательные объекты симметрично уравновесили друг друга.


Сделаем один шаг

$$
\beta^{(1)}=\beta^{(0)}-\eta\nabla R(\beta^{(0)}),
\qquad \eta=0.5.
$$


In [ ]:
eta_demo = 0.5
beta_after_step = beta_start - eta_demo * grad_R_start

pd.DataFrame({
    "параметр": ["beta0", "beta1", "beta2"],
    "до шага": beta_start,
    "после шага": beta_after_step,
})

Пересчитаем score и вероятности с новыми параметрами.


Теперь это уже новый вектор параметров, поэтому пересчитываем всю цепочку `score → sigmoid`.

In [ ]:
z_after_step = X_small @ beta_after_step
p_after_step = sigmoid(z_after_step)

pd.DataFrame({
    "y": y_small,
    "score до": z_start,
    "score после": z_after_step,
    "p до": p_start,
    "p после": p_after_step,
})

Сравним среднюю log loss до и после шага.


In [ ]:
loss_start = binary_log_loss(y_small, p_start)
loss_after_step = binary_log_loss(y_small, p_after_step)

print("До шага:", round(loss_start, 6))
print("После шага:", round(loss_after_step, 6))

На этом примере один шаг уменьшил среднюю log loss и сдвинул вероятности в сторону наблюдаемых классов. На больших выборках отдельные объекты могут двигаться в разные стороны: один общий вектор параметров минимизирует среднюю целевую функцию по всей обучающей выборке.


## 4. Градиент для всей выборки

Только что мы по объектам получили

$$
\nabla R(\beta)
=
\frac1n\sum_{i=1}^n(p_i-y_i)x_i.
$$

Та же сумма записывается компактно:

$$
\nabla R(\beta)
=
\frac1nX^\top(p-y).
$$

Теперь реализуем эту формулу сразу для всей выборки.


### Задание 3. Реализуйте `logistic_gradient`


In [ ]:
def logistic_gradient(beta, X, y):
    """
    Вычисляет градиент среднего log loss по параметрам логистической регрессии.

    Parameters
    ----------
    beta : array-like, shape (d,)
        Текущий вектор параметров модели.
    X : array-like, shape (n, d)
        Матрица признаков. Если свободный член включён в beta,
        первый столбец X должен состоять из единиц.
    y : array-like, shape (n,)
        Истинные бинарные метки 0/1.

    Returns
    -------
    grad : np.ndarray, shape (d,)
        Градиент среднего log loss:
        по одной частной производной на каждый параметр beta.
    """
    beta = np.asarray(beta, dtype=float)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    ...

    return grad


Проверка здесь имеет конкретный смысл: компактная функция должна вернуть тот же градиент, который мы только что собрали по четырём объектам вручную.


In [ ]:
grad_from_function = logistic_gradient(beta_start, X_small, y_small)

pd.DataFrame({
    "параметр": ["beta0", "beta1", "beta2"],
    "по объектам вручную": grad_R_start,
    "logistic_gradient": grad_from_function,
})

In [ ]:
print("Совпадают:", np.allclose(grad_R_start, grad_from_function))
assert np.allclose(grad_R_start, grad_from_function)

## 5. Обучаем модель

Для градиентного спуска нужны две функции:

- целевая функция $R(\beta)$;
- её градиент $\nabla R(\beta)$.

Градиент мы уже написали. Целевая функция собирается из тех же шагов: score → sigmoid → log loss.


In [ ]:
def logistic_objective(beta, X, y):
    """Вычисляет среднюю log loss для текущего вектора параметров beta."""
    z = X @ beta
    p = sigmoid(z)
    return binary_log_loss(y, p)


При нулевых параметрах модель выдаёт вероятность 0.5 каждому объекту. Посмотрим на начальное значение целевой функции.


In [ ]:
beta_start = np.zeros(X_train.shape[1])
initial_loss = logistic_objective(beta_start, X_train, y_train)

print("Начальная log loss:", round(initial_loss, 6))


Ниже стоит **та же функция `gradient_descent` из семинара 3**. Сама функция остаётся без изменений. Для логистической регрессии передаём ей новую целевую функцию и новый градиент.


In [ ]:
def gradient_descent(
    theta0,
    objective_fn,
    grad_fn,
    learning_rate,
    n_steps,
    fixed_args=(),
):
    """
    Минимизирует целевую функцию градиентным спуском с постоянным шагом.

    Parameters
    ----------
    theta0 : array-like, shape (d,)
        Начальное значение вектора из d параметров.
    objective_fn : callable
        Целевая функция. Первый аргумент — theta; после него при
        необходимости передаются элементы fixed_args.
    grad_fn : callable
        Функция градиента. Возвращает одномерный массив длины d:
        по одной частной производной на каждый параметр theta.
    learning_rate : float
        Положительный темп обучения eta.
    n_steps : int
        Число обновлений параметров.
    fixed_args : tuple, default=()
        Дополнительные неизменяемые аргументы для objective_fn и grad_fn.
        Например, для линейной регрессии это может быть (X, y).

    Returns
    -------
    theta : np.ndarray, shape (d,)
        Вектор параметров после последнего обновления.
    history : dict[str, np.ndarray]
        История параметров и значений целевой функции.
        history["theta"] имеет размер (n_steps + 1, d),
        history["objective"] — длину n_steps + 1.
    """
    theta = np.array(theta0, dtype=float, copy=True)
    history = {
        "theta": [theta.copy()],
        "objective": [float(objective_fn(theta, *fixed_args))],
    }

    for _ in range(n_steps):
        # 1. Локальная информация в текущей точке.
        grad = np.array(grad_fn(theta, *fixed_args), dtype=float)

        # 2. Шаг по антиградиенту.
        theta = theta - learning_rate * grad

        # 3. Сохраняем уже обновлённое состояние.
        history["theta"].append(theta.copy())
        history["objective"].append(float(objective_fn(theta, *fixed_args)))

    history["theta"] = np.asarray(history["theta"])
    history["objective"] = np.asarray(history["objective"])
    return theta, history


Запустим обучение на обучающей части данных.


In [ ]:
beta_hat, history = gradient_descent(
    theta0=beta_start,
    objective_fn=logistic_objective,
    grad_fn=logistic_gradient,
    learning_rate=0.08,
    n_steps=2000,
    fixed_args=(X_train, y_train),
)


Посмотрим на найденные коэффициенты.


In [ ]:
pd.Series(
    beta_hat,
    index=["intercept", *feature_names],
    name="beta_hat",
)


Сравним значение целевой функции в начале и в конце обучения.


In [ ]:
print("Начало:", round(history["objective"][0], 6))
print("Конец:", round(history["objective"][-1], 6))


Посмотрим и на диапазон score, который получился на нашей обучающей выборке. Это возвращает нас к практической оговорке из раздела про сигмоиду.


In [ ]:
score_train = X_train @ beta_hat

print("Минимальный score:", round(score_train.min(), 3))
print("Максимальный score:", round(score_train.max(), 3))


Здесь score остаются умеренными по величине, поэтому обычная формула сигмоиды вычисляется без переполнения. На другом наборе данных диапазон может быть шире; общая численная реализация должна учитывать и такие случаи.


График показывает всю историю оптимизации.


In [ ]:
plt.figure(figsize=(7.2, 4.2))
plt.plot(history["objective"])
plt.xlabel("Шаг градиентного спуска")
plt.ylabel("Средняя log loss")
plt.show()


In [ ]:
assert history["objective"][-1] < history["objective"][0]


## 6. Вероятности после обучения

После обучения у нас есть $\hat\beta$. Для бинарной задачи сначала вычисляем

$$
\hat p=P(Y=1\mid X=x)=\sigma(x^\top\hat\beta).
$$

Вероятность второго класса автоматически равна $1-\hat p$.

Библиотечный интерфейс `predict_proba` обычно возвращает **вероятности всех классов**, поэтому наша функция будет возвращать матрицу размера $n\times2$:

$$
\begin{bmatrix}
P(Y=0\mid x_1) & P(Y=1\mid x_1)\\
\vdots & \vdots\\
P(Y=0\mid x_n) & P(Y=1\mid x_n)
\end{bmatrix}.
$$

Для меток 0/1 второй столбец — та самая вероятность положительного класса, с которой мы работали на лекции.


### Задание 4. Реализуйте `predict_proba`


In [ ]:
def predict_proba(X, beta):
    """
    Вычисляет вероятности обоих классов для каждого объекта.

    Parameters
    ----------
    X : array-like, shape (n, d)
        Матрица признаков модели.
    beta : array-like, shape (d,)
        Вектор обученных параметров.

    Returns
    -------
    proba : np.ndarray, shape (n, 2)
        Вероятности классов для каждого объекта.
        Первый столбец содержит P(Y=0 | X=x),
        второй — P(Y=1 | X=x).
        В каждой строке вероятности суммируются в 1.
    """
    X = np.asarray(X, dtype=float)
    beta = np.asarray(beta, dtype=float)

    ...

    return np.column_stack([1.0 - p1, p1])


Получим вероятности для валидационной части.


In [ ]:
proba_valid_manual = predict_proba(X_valid, beta_hat)
p_valid_manual = proba_valid_manual[:, 1]

valid_predictions_manual = X_valid_df.copy()
valid_predictions_manual["y"] = y_valid_series.to_numpy()
valid_predictions_manual["P(Y=0)"] = proba_valid_manual[:, 0]
valid_predictions_manual["P(Y=1)"] = proba_valid_manual[:, 1]

valid_predictions_manual.head(8)

In [ ]:
assert proba_valid_manual.shape == (len(y_valid), 2)
assert np.all((proba_valid_manual >= 0.0) & (proba_valid_manual <= 1.0))
assert np.allclose(proba_valid_manual.sum(axis=1), 1.0)

## 7. Один объект целиком: признаки → score → вероятность → решение

Теперь соберём все уровни результата на одном клиенте.

Выберем объект, для которого модель дала вероятность около 0.65. Это удобно для сравнения двух порогов.


In [ ]:
target_probability = 0.65
row_pos = int(np.argmin(np.abs(p_valid_manual - target_probability)))

x_one_df = X_valid_df.iloc[[row_pos]]
x_one = X_valid[row_pos]
y_one = int(y_valid[row_pos])

x_one_df.assign(y=y_one)

Сначала вычисляется линейный score.


In [ ]:
score_one = float(x_one @ beta_hat)
score_one


Теперь получим обе вероятности через нашу `predict_proba`.

Здесь появляется конструкция

```python
x_one.reshape(1, -1)
```

`x_one` — один объект, поэтому после выбора строки он имеет форму `(d,)`. Но `predict_proba` написана для **матрицы объектов** формы `(n, d)`.

`reshape(1, -1)` превращает вектор `(d,)` в матрицу с одной строкой `(1, d)`:

- `1` означает «один объект»;
- `-1` означает «число столбцов NumPy выведет сам».

Такой `reshape` периодически встречается, когда функция ожидает batch/матрицу, а у нас на руках один объект.

`predict_proba(...)` вернёт форму `(1, 2)`, а финальное `[0]` достаёт из неё единственную строку с двумя вероятностями.

In [ ]:
proba_one = predict_proba(x_one.reshape(1, -1), beta_hat)[0]
p_one = float(proba_one[1])

pd.Series({
    "P(Y=0 | x)": proba_one[0],
    "P(Y=1 | x)": proba_one[1],
})


Порог применяется уже к готовой вероятности.


In [ ]:
decision_05 = int(p_one >= 0.5)
decision_08 = int(p_one >= 0.8)

pd.DataFrame({
    "величина": ["score", "вероятность класса 1", "класс при t=0.5", "класс при t=0.8"],
    "значение": [score_one, p_one, decision_05, decision_08],
})


Здесь хорошо видно разделение уровней:

$$
x
\longrightarrow
\hat z
\longrightarrow
\hat p
\overset{t}{\longrightarrow}
\hat y.
$$

При смене порога score и вероятность этого клиента остаются прежними. Меняется правило, которое превращает вероятность в метку класса.


## 8. Сверяем ручную модель со scikit-learn

Для сравнения обучим `LogisticRegression` без регуляризации. Библиотечной модели передадим исходные два признака; свободный член она добавит сама.

`clf.fit` получает ровно те же обучающие строки `X_train_df, y_train_series`, на которых мы обучали ручную модель, а ниже обе модели сравниваются на том же `X_valid_df`.

Укажем

```python
solver="lbfgs"
```

`solver` — это алгоритм, который численно подбирает коэффициенты, минимизируя log loss.

**L-BFGS** (`Limited-memory BFGS`) — квазиньютоновский метод: он использует градиенты и по истории нескольких последних шагов приближённо восстанавливает информацию о кривизне функции. Поэтому ему не нужно задавать наш фиксированный `learning_rate`, как обычному gradient descent.

Для семинара достаточно помнить: **модель и целевая функция те же, способ численной оптимизации другой**. `max_iter=5000` — только верхняя граница числа итераций; solver может остановиться раньше по своему критерию сходимости.

In [ ]:
clf = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
)


In [ ]:
clf.fit(X_train_df, y_train_series)


Сначала сравним линейные score.


In [ ]:
score_valid_manual = X_valid @ beta_hat
score_valid_sklearn = clf.decision_function(X_valid_df)

score_comparison = pd.DataFrame(
    {
        "manual score": score_valid_manual,
        "sklearn score": score_valid_sklearn,
    },
    index=X_valid_df.index,
)
score_comparison["difference"] = (
    score_comparison["manual score"] - score_comparison["sklearn score"]
)

score_comparison.head(8)

Теперь сравним `predict_proba`. В scikit-learn столбцы идут в порядке `clf.classes_`.


In [ ]:
proba_valid_sklearn = clf.predict_proba(X_valid_df)

print("Порядок столбцов sklearn:", clf.classes_)

proba_comparison = pd.DataFrame(
    {
        "manual P(Y=0)": proba_valid_manual[:, 0],
        "manual P(Y=1)": proba_valid_manual[:, 1],
        "sklearn P(Y=0)": proba_valid_sklearn[:, 0],
        "sklearn P(Y=1)": proba_valid_sklearn[:, 1],
    },
    index=X_valid_df.index,
)
proba_comparison["|delta P(Y=1)|"] = np.abs(
    proba_comparison["manual P(Y=1)"] - proba_comparison["sklearn P(Y=1)"]
)

proba_comparison.head(8)

In [ ]:
print(
    "Максимальное отличие score:",
    np.max(np.abs(score_valid_manual - score_valid_sklearn)),
)
print(
    "Максимальное отличие вероятностей:",
    np.max(np.abs(proba_valid_manual - proba_valid_sklearn)),
)

Сравним и сами коэффициенты.


In [ ]:
beta_sklearn = np.r_[clf.intercept_, clf.coef_.ravel()]

pd.DataFrame({
    "параметр": ["intercept", *feature_names],
    "ручная модель": beta_hat,
    "sklearn": beta_sklearn,
})


Связь с интерфейсом библиотеки теперь можно читать буквально:

- `decision_function(X)` возвращает линейный score $X\hat\beta$;
- `predict_proba(X)` возвращает по строке вероятностей для всех классов;
- при `clf.classes_ == [0, 1]` второй столбец `predict_proba(X)[:, 1]` равен $\sigma(X\hat\beta)$;
- `predict(X)` превращает вероятности в жёсткие метки по правилу решения модели.

Во всех сравнительных таблицах сохранён тот же `row_id`, поэтому строка `53` — это один и тот же validation-объект и в ручной модели, и в scikit-learn.

Небольшие различия вероятностей ожидаемы: наша ручная модель сделала ровно 2000 шагов gradient descent с фиксированным `learning_rate=0.08`, а `lbfgs` шёл к минимуму своим способом и остановился по собственному критерию сходимости. При `penalty=None` обе модели минимизируют одну и ту же нерегуляризованную log loss на **одной и той же train-части**, поэтому результаты должны быть очень близкими, но не обязаны совпадать бит-в-бит.

## 9. Линейная граница решения и вероятности

Вернёмся в пространство исходных признаков $(x_1,x_2)$.

Логистическая регрессия сначала строит **линейный score**

$$
z(x)=\hat\beta_0+\hat\beta_1x_1+\hat\beta_2x_2.
$$

При пороге $t=0.5$

$$
\hat p(x)\ge0.5
\Longleftrightarrow
z(x)\ge0.
$$

Поэтому граница решения задаётся условием

$$
\hat\beta_0+\hat\beta_1x_1+\hat\beta_2x_2=0.
$$

В пространстве двух признаков это прямая; в пространстве $d$ признаков — гиперплоскость. Она делит пространство признаков на две области решения: $z<0$ и $z>0$. Данные при этом не обязаны быть идеально разделимы этой прямой.


Сначала вычислим координаты этой прямой.


In [ ]:
x1_grid = np.linspace(
    X_train_df["weekly_sessions_change"].min() - 0.5,
    X_train_df["weekly_sessions_change"].max() + 0.5,
    200,
)

x2_boundary = -(beta_hat[0] + beta_hat[1] * x1_grid) / beta_hat[2]


Теперь нанесём границу на ту же плоскость признаков.


In [ ]:
plt.figure(figsize=(7.4, 5.0))

for cls, marker, label in [
    (0, "o", "класс 0"),
    (1, "^", "класс 1"),
]:
    mask = y_train == cls
    plt.scatter(
        X_train_features[mask, 0],
        X_train_features[mask, 1],
        marker=marker,
        label=label,
        alpha=0.8,
    )

plt.plot(x1_grid, x2_boundary, label="граница: score = 0")
plt.xlabel("Изменение числа сессий за неделю")
plt.ylabel("Обращения в поддержку за 30 дней")
plt.legend()
plt.show()


На этом графике мы смотрим именно на **пространство признаков**. Прямая $z(x)=0$ задаёт границу при пороге 0.5:

- по одну сторону $z<0$, поэтому $P(Y=1\mid x)<0.5$;
- на самой границе $z=0$, поэтому $P(Y=1\mid x)=0.5$;
- по другую сторону $z>0$, поэтому $P(Y=1\mid x)>0.5$.

Сигмоида не искривляет эту границу. Она берёт уже посчитанный одномерный score и переводит его на вероятностную шкалу.


Теперь покажем **те же обучающие объекты на оси score**. Два исходных признака уже сведены к одному числу $z=x^\top\hat\beta$, а сигмоида переводит это число в вероятность.

In [ ]:
score_train = X_train @ beta_hat
p_train = predict_proba(X_train, beta_hat)[:, 1]

z_curve = np.linspace(score_train.min() - 1.0, score_train.max() + 1.0, 300)

plt.figure(figsize=(7.4, 5.0))
plt.plot(z_curve, sigmoid(z_curve), label="p = sigmoid(z)")

for cls, marker, label in [
    (0, "o", "истинный класс 0"),
    (1, "^", "истинный класс 1"),
]:
    mask = y_train == cls
    plt.scatter(
        score_train[mask],
        p_train[mask],
        marker=marker,
        alpha=0.8,
        label=label,
    )

plt.axvline(0.0, linestyle="--", label="score = 0")
plt.axhline(0.5, linestyle="--", label="p = 0.5")
plt.xlabel("Линейный score z")
plt.ylabel("P(Y=1 | x)")
plt.legend()
plt.show()

Здесь уже нет двумерного пространства признаков: каждому объекту соответствует одно значение score. Сигмоида монотонна, поэтому объекты сохраняют порядок на этой оси. Точка $z=0$ переходит в $p=0.5$.

Получаются две совместимые картины:

- **в пространстве признаков** решение задаётся линейной гиперплоскостью;
- **на шкале score** сигмоида даёт вероятностную интерпретацию положения объекта относительно этой границы.

Для другого порога $t$ граница задавалась бы условием $z=\operatorname{logit}(t)$ и всё равно оставалась бы гиперплоскостью.

## 10. Один прогноз вероятности — разные решения

Дальше снова используем вероятности **нашей ручной модели**, уже рассчитанные в разделе 6:

```python
p_valid_manual
```

Блок со scikit-learn выше был только сверкой реализации. Новый split и новые вероятности для этого упражнения не создаём.

Сравним пороги 0.5 и 0.3.

In [ ]:
pred_05 = (p_valid_manual >= 0.5).astype(int)
pred_03 = (p_valid_manual >= 0.3).astype(int)

Посмотрим прежде всего на объекты, для которых два порога дали разные метки.

Это **не новая validation-выборка** и вероятности здесь не пересчитываются. Мы берём ту же таблицу `valid_predictions_manual`, затем:

1. оставляем только строки, где решения при `t=0.5` и `t=0.3` различаются;
2. сортируем эти строки по `P(Y=1)`.

Поэтому набор строк и их порядок отличаются от `valid_predictions_manual.head(8)`, но `row_id` позволяет проверить любой конкретный объект.

In [ ]:
threshold_table = valid_predictions_manual.copy()
threshold_table["pred_t_0.5"] = pred_05
threshold_table["pred_t_0.3"] = pred_03

threshold_disagreements = threshold_table[
    threshold_table["pred_t_0.5"] != threshold_table["pred_t_0.3"]
].sort_values("P(Y=1)")

threshold_disagreements

Посчитайте accuracy для двух правил решения.


In [ ]:
accuracy_05 = np.mean(pred_05 == y_valid)
accuracy_03 = np.mean(pred_03 == y_valid)

pd.DataFrame({
    "threshold": [0.5, 0.3],
    "accuracy": [accuracy_05, accuracy_03],
})


### Вопросы для обсуждения

1. Изменился ли вектор `beta_hat` после смены порога?
2. Изменились ли `p_valid_manual`?
3. Какие именно `row_id` получили другую метку?
4. Всегда ли порог с большей accuracy является лучшим решением для прикладной задачи?

Последний вопрос останется открытым до следующей недели: там появятся разные типы ошибок и метрики классификации.

## 11. Итог

Мы собрали логистическую регрессию от вычисления score до итогового решения.

Во время обучения:

$$
X
\longrightarrow
z=X\beta
\longrightarrow
p=\sigma(z)
\longrightarrow
\text{log loss}
\longrightarrow
\nabla R(\beta)=\frac1nX^\top(p-y)
\longrightarrow
\hat\beta.
$$

После обучения один объект проходит другой короткий путь:

$$
x
\longrightarrow
\hat z
\longrightarrow
\bigl(P(Y=0\mid x),P(Y=1\mid x)\bigr)
\overset{\text{порог}}{\longrightarrow}
\hat y.
$$

Что важно вынести из семинара:

- логистическая регрессия остаётся **линейным классификатором по геометрии решения**: при фиксированном пороге пространство признаков делится гиперплоскостью;
- сигмоида не меняет эту линейную геометрию, а переводит линейный score на вероятностную шкалу;
- Bernoulli log loss задаёт критерий обучения, а её градиент имеет компактный вид $\frac1nX^\top(p-y)$;
- для оптимизации подошёл тот же общий `gradient_descent`, который использовался на семинаре 3;
- `predict_proba` возвращает вероятности классов, а выбор порога относится уже к правилу принятия решения;
- изменение порога не меняет обученные коэффициенты и вероятности — меняется только итоговая метка класса.

Ручная реализация и scikit-learn описывают одни и те же уровни модели: `decision_function` — score, `predict_proba` — вероятности, `predict` — решение.
